# Stage 1 — Fine-tune ViT on PETA

Trains a `google/vit-base-patch16-224` backbone for binary gender classification.

**Backbone**: pure ViT (not CLIP).


## 0. Setup

In [ ]:
!pip install -q opencv-python-headless ultralytics transformers h5py scipy \
                pyyaml matplotlib huggingface_hub

import torch, os, json, time, math, csv, shutil, sys
from pathlib import Path
from collections import defaultdict
import numpy as np
from PIL import Image
import cv2

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, 'cuda:', torch.cuda.is_available())
print('Using device:', DEVICE)


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
DRIVE_CKPT_DIR = '/content/drive/MyDrive/IPD_checkpoints'
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
print('Checkpoints will be saved to:', DRIVE_CKPT_DIR)


## 1. Config

In [ ]:
CONFIG = {
    'backbone': 'google/vit-base-patch16-224',
    'seed': 42,
    'peta_csv':    '/content/data/PETA/labels.csv',
    'peta_imgs':   '/content/data/PETA/images',
    'val_ratio': 0.1,
    'stage1': {
        'output_dir': '/content/checkpoints/stage1',
        'phase_a_epochs': 4, 'phase_a_head_lr': 1e-3,
        'phase_b_epochs': 10, 'phase_b_head_lr': 1e-5, 'phase_b_backbone_lr': 1e-5,
        'batch_size': 64, 'eval_batch_size': 128, 'num_workers': 4,
        'weight_decay': 0.3, 'warmup_ratio': 0.2, 'grad_clip': 1.0,
        'fp16': True, 'log_every': 50,
    },
    'hf_repo': 'abhshkp/footfall-analysis-vit-stage1',
}
torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
print('Config loaded. Backbone:', CONFIG['backbone'])
print('HF repo:', CONFIG['hf_repo'])


## 2. Helper functions

In [ ]:
from transformers import ViTImageProcessor
_VIT_PROC = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224')

def _to_tensor(pil_img):
    out = _VIT_PROC(images=pil_img, return_tensors='pt')
    return out['pixel_values'][0]

def train_transform(pil_img):
    if torch.rand(1).item() < 0.5:
        pil_img = pil_img.transpose(Image.FLIP_LEFT_RIGHT)
    from PIL import ImageEnhance
    pil_img = ImageEnhance.Brightness(pil_img).enhance(0.9 + 0.1*torch.rand(1).item())
    pil_img = ImageEnhance.Contrast(pil_img).enhance(0.9 + 0.1*torch.rand(1).item())
    pil_img = ImageEnhance.Color(pil_img).enhance(0.9 + 0.1*torch.rand(1).item())
    return _to_tensor(pil_img)

def val_transform(pil_img):
    return _to_tensor(pil_img)

print('Transforms ready.')

In [ ]:
from torch.utils.data import Dataset, DataLoader, Subset

GENDER_TO_IDX = {'female': 0, 'male': 1}
IDX_TO_GENDER = {0: 'female', 1: 'male'}

class CSVDataset(Dataset):
    def __init__(self, csv_path, image_root, transform=None):
        self.image_root = Path(image_root); self.transform = transform; self.rows = []
        with open(csv_path) as fh:
            for r in csv.DictReader(fh):
                g = r['gender'].strip().lower()
                if g not in GENDER_TO_IDX: continue
                p = Path(r['image'])
                if not p.is_absolute(): p = self.image_root / p
                if not p.exists(): continue
                self.rows.append((p, GENDER_TO_IDX[g]))
        if not self.rows: raise RuntimeError(f'No valid rows in {csv_path}')
    def __len__(self): return len(self.rows)
    def __getitem__(self, idx):
        p, lab = self.rows[idx]
        img = Image.open(p).convert('RGB')
        if self.transform is not None: img = self.transform(img)
        return img, lab

class FolderPerClassDataset(Dataset):
    def __init__(self, root, transform=None, exts=('.jpg', '.jpeg', '.png')):
        self.root = Path(root); self.transform = transform; self.rows = []
        for name, idx in GENDER_TO_IDX.items():
            d = self.root / name
            if not d.exists(): continue
            for ext in exts:
                for img in d.glob(f'*{ext}'): self.rows.append((img, idx))
                for img in d.glob(f'*{ext.upper()}'): self.rows.append((img, idx))
        if not self.rows: raise RuntimeError(f'No images in {self.root}/{{male,female}}/')
    def __len__(self): return len(self.rows)
    def __getitem__(self, idx):
        p, lab = self.rows[idx]
        img = Image.open(p).convert('RGB')
        if self.transform is not None: img = self.transform(img)
        return img, lab

class ConcatDataset(Dataset):
    def __init__(self, datasets):
        self.datasets = datasets; self.cum = [0]
        for d in datasets: self.cum.append(self.cum[-1] + len(d))
    def __len__(self): return self.cum[-1]
    def __getitem__(self, idx):
        for i, c in enumerate(self.cum[1:]):
            if idx < c: return self.datasets[i][idx - self.cum[i]]
        raise IndexError(idx)

def stratified_split(rows, val_ratio=0.1, seed=42):
    g = torch.Generator().manual_seed(seed)
    by_class = {}
    for i, (_, lab) in enumerate(rows): by_class.setdefault(lab, []).append(i)
    train_idx, val_idx = [], []
    for lab, idxs in by_class.items():
        perm = torch.randperm(len(idxs), generator=g).tolist()
        n_val = int(len(idxs) * val_ratio)
        for j, p in enumerate(perm):
            (val_idx if j < n_val else train_idx).append(idxs[p])
    return train_idx, val_idx

print('Datasets ready.')

In [ ]:
import torch.nn as nn
from transformers import ViTModel

class GenderClassifier(nn.Module):
    def __init__(self, vit_model_name='google/vit-base-patch16-224', num_labels=2):
        super().__init__()
        self.vit = ViTModel.from_pretrained(vit_model_name)
        hidden_dim = self.vit.config.hidden_size  # 768
        self.head = nn.Linear(hidden_dim, num_labels)
        self.num_labels = num_labels
        self.id2label = {0: 'female', 1: 'male'}
        self.label2id = {'female': 0, 'male': 1}
        self.vit_model_name = vit_model_name

    def forward(self, pixel_values):
        out = self.vit(pixel_values=pixel_values)
        return self.head(out.pooler_output)

    def freeze_backbone(self):
        for p in self.vit.parameters(): p.requires_grad = False
        for p in self.head.parameters(): p.requires_grad = True

    def unfreeze_backbone(self):
        for p in self.parameters(): p.requires_grad = True

    def param_groups(self, head_lr, backbone_lr):
        head_p, backbone_p = [], []
        for name, p in self.named_parameters():
            if not p.requires_grad: continue
            (head_p if name.startswith('head.') else backbone_p).append(p)
        groups = []
        if head_p: groups.append({'params': head_p, 'lr': head_lr, 'name': 'head'})
        if backbone_p: groups.append({'params': backbone_p, 'lr': backbone_lr, 'name': 'backbone'})
        return groups

    def save(self, out_dir):
        out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
        torch.save(self.state_dict(), out_dir / 'pytorch_model.bin')
        cfg = {'architectures': ['GenderClassifier'], 'model_type': 'vit_gender',
               'vit_model_name': self.vit_model_name, 'num_labels': self.num_labels,
               'id2label': self.id2label, 'label2id': self.label2id}
        with open(out_dir / 'config.json', 'w') as f: json.dump(cfg, f, indent=2)
        print(f'Saved checkpoint to {out_dir}')

    @classmethod
    def load(cls, ckpt_dir, device='cpu'):
        with open(Path(ckpt_dir) / 'config.json') as f: cfg = json.load(f)
        m = cls(cfg['vit_model_name'], cfg.get('num_labels', 2))
        state = torch.load(Path(ckpt_dir) / 'pytorch_model.bin', map_location=device)
        m.load_state_dict(state, strict=False)
        m.to(device).eval()
        return m

print('Model class ready.')

In [ ]:
def _cosine_with_warmup(opt, warmup, total):
    from torch.optim.lr_scheduler import LambdaLR
    def lam(step):
        if step < warmup: return step / max(1, warmup)
        prog = (step - warmup) / max(1, total - warmup)
        return max(0.0, 0.5 * (1 + math.cos(math.pi * prog)))
    return LambdaLR(opt, lam)

def evaluate(model, loader, device):
    model.eval(); nc, nt, mc, mt, fc, ft = 0, 0, 0, 0, 0, 0
    with torch.no_grad():
        for imgs, labs in loader:
            imgs, labs = imgs.to(device), labs.to(device)
            with torch.amp.autocast('cuda', enabled=(device=='cuda')):
                logits = model(imgs)
            preds = logits.argmax(-1)
            nc += int((preds==labs).sum()); nt += labs.size(0)
            for l, p in zip(labs.tolist(), preds.tolist()):
                if l == 1: mt += 1; mc += int(p==1)
                else: ft += 1; fc += int(p==0)
    return {'accuracy': nc/max(nt,1), 'male_acc': mc/max(mt,1),
            'female_acc': fc/max(ft,1), 'n_total': nt}

def train_phase(model, train_loader, val_loader, device, phase_name,
                epochs, head_lr, backbone_lr, freeze,
                weight_decay=0.01, warmup_ratio=0.1, grad_clip=1.0,
                fp16=True, log_every=50, output_dir='.', best_acc=0.0):
    if freeze:
        model.freeze_backbone(); print(f'[{phase_name}] backbone FROZEN, head only')
    else:
        model.unfreeze_backbone(); print(f'[{phase_name}] backbone UNFROZEN, full fine-tune')
    opt = torch.optim.AdamW(model.param_groups(head_lr, backbone_lr), weight_decay=weight_decay)
    total_steps = epochs * len(train_loader)
    warmup = int(warmup_ratio * total_steps)
    sched = _cosine_with_warmup(opt, warmup, total_steps)
    scaler = torch.amp.GradScaler('cuda', enabled=(fp16 and device=='cuda'))
    step, t0, history = 0, time.time(), []
    for ep in range(epochs):
        model.train(); rloss, rcorr, rtot = 0.0, 0, 0
        for imgs, labs in train_loader:
            imgs, labs = imgs.to(device), labs.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=(fp16 and device=='cuda')):
                logits = model(imgs)
                loss = nn.functional.cross_entropy(logits, labs)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(opt); scaler.update(); sched.step(); step += 1
            rloss += loss.item()*imgs.size(0)
            rcorr += int((logits.argmax(-1)==labs).sum()); rtot += imgs.size(0)
            if step % log_every == 0:
                lr = opt.param_groups[0]['lr']
                eta = (time.time()-t0)/step*(total_steps-step)
                print(f'  [{phase_name}] ep{ep+1}/{epochs} step {step}/{total_steps} '
                      f'loss={rloss/rtot:.4f} acc={rcorr/rtot:.3f} lr={lr:.2e} eta={eta:.0f}s')
        m = evaluate(model, val_loader, device)
        print(f'  [{phase_name}] ep{ep+1} VAL acc={m["accuracy"]:.4f} '
              f'male={m["male_acc"]:.3f} female={m["female_acc"]:.3f} n={m["n_total"]}')
        history.append({'phase': phase_name, 'epoch': ep+1, **m})
        if m['accuracy'] > best_acc:
            best_acc = m['accuracy']
            print(f'  [{phase_name}] NEW BEST, saving to {output_dir}/best')
            model.save(os.path.join(output_dir, 'best'))
    return best_acc, history

print('Trainer ready.')

## 3. Datasets — PETA

Upload the PETA dataset raw zip, then run the organize cell.
Download link: see `scripts/download_datasets.md` in the repo.


In [ ]:
from google.colab import files
print('Upload PETA.zip')
uploaded = files.upload()
for name in uploaded.keys():
    os.makedirs('/content/data/raw', exist_ok=True)
    import zipfile
    with zipfile.ZipFile(name) as zf: zf.extractall('/content/data/raw/')
    print(f'  extracted {name} to /content/data/raw/')


In [ ]:
# === Organize PETA ===
# Tries PETA.mat (HDF5) first; falls back to Label.txt files.
def organize_peta(raw_dir='/content/data/raw', out_dir='/content/data/PETA'):
    import re
    raw = Path(raw_dir)
    (Path(out_dir)/'images').mkdir(parents=True, exist_ok=True)
    labels, img_exts = [], ('.png', '.bmp', '.jpg', '.jpeg')
    gender_pat = re.compile(r'female|male', re.IGNORECASE)
    label_files = list(raw.rglob('Label.txt'))
    if label_files:
        print(f'[peta] found {len(label_files)} Label.txt files')
        for lf in label_files:
            d = lf.parent
            with open(lf) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) < 2: continue
                    img_name = parts[0]
                    rest = ' '.join(parts[1:]).lower()
                    m = gender_pat.search(rest)
                    if not m: continue
                    gender = m.group(0).lower()
                    src = d / img_name
                    if not src.exists():
                        for ext in img_exts:
                            cand = d / (img_name + ext)
                            if cand.exists(): src = cand; break
                    if not src.exists(): continue
                    dst = Path(out_dir)/'images'/src.name
                    if not dst.exists(): shutil.copy2(src, dst)
                    labels.append({'image': src.name, 'gender': gender})
    else:
        import h5py
        mat_files = list(raw.rglob('PETA*.mat'))
        if not mat_files:
            print('No PETA*.mat or Label.txt found under', raw); return
        mat_path = mat_files[0]
        print(f'[peta] reading {mat_path}')
        with h5py.File(mat_path, 'r') as f:
            root_key = list(f.keys())[0]; root = f[root_key]
            attr_names_ref = root['attrs']['name'][:]
            attr_values = root['attrs']['value'][:]
            image_names_ref = root['image']['name'][:]
            attr_names = [''.join(chr(c[0]) for c in f[ref][:]).strip() for ref in attr_names_ref]
            image_names = [''.join(chr(c[0]) for c in f[ref][:]).strip() for ref in image_names_ref.flat]
            if attr_values.shape[0] == len(attr_names): attr_values = attr_values.T
            try: male_idx = attr_names.index('personalMale')
            except ValueError: male_idx = next(i for i, n in enumerate(attr_names) if 'male' in n.lower() and 'fe' not in n.lower())
        image_dirs = [p for p in raw.rglob('*') if p.is_dir() and any(p.glob('*.jpg'))]
        name_to_path = {img.name.lower(): img for d in image_dirs for img in d.glob('*.jpg')}
        for i, name in enumerate(image_names):
            src = name_to_path.get(name.lower())
            if src is None: continue
            dst = Path(out_dir)/'images'/name
            if not dst.exists(): shutil.copy2(src, dst)
            labels.append({'image': name, 'gender': 'male' if int(attr_values[i, male_idx])==1 else 'female'})
    with open(Path(out_dir)/'labels.csv', 'w', newline='') as fh:
        w = csv.DictWriter(fh, fieldnames=['image','gender']); w.writeheader(); w.writerows(labels)
    nm = sum(1 for r in labels if r['gender']=='male')
    nf = sum(1 for r in labels if r['gender']=='female')
    print(f'[peta] wrote {len(labels)} images. male={nm} female={nf}')

organize_peta()


In [ ]:
# Verify PETA dataset
for name, csv_path, img_root in [
    ('PETA',   CONFIG['peta_csv'],   CONFIG['peta_imgs']),
]:
    if Path(csv_path).exists():
        n = sum(1 for _ in open(csv_path)) - 1
        n_imgs = len(list(Path(img_root).glob('*.jpg'))) if Path(img_root).exists() else 0
        print(f'  {name:7s}: {n} labels, {n_imgs} images')
    else:
        print(f'  {name:7s}: MISSING ({csv_path})')


## 4. Train Stage 1

In [ ]:
s1 = CONFIG['stage1']
csv_p = CONFIG['peta_csv']
img_root = CONFIG['peta_imgs']
if not Path(csv_p).exists():
    raise RuntimeError(f'PETA dataset not found at {csv_p}')

full_val = CSVDataset(csv_p, img_root, transform=val_transform)
full_train = CSVDataset(csv_p, img_root, transform=train_transform)
tr_idx, va_idx = stratified_split(full_val.rows, CONFIG['val_ratio'], CONFIG['seed'])
train_dset = Subset(full_train, tr_idx)
val_dset = Subset(full_val, va_idx)
print(f'  PETA: train={len(tr_idx)} val={len(va_idx)}')

train_loader = DataLoader(train_dset, batch_size=s1['batch_size'], shuffle=True,
                          num_workers=s1['num_workers'], pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dset, batch_size=s1['eval_batch_size'], shuffle=False,
                        num_workers=s1['num_workers'], pin_memory=True)
print('Dataloaders ready.')


In [ ]:
model = GenderClassifier(CONFIG['backbone'])
os.makedirs(s1['output_dir'], exist_ok=True)
best_acc = 0.0
all_history = []

# Phase A: linear probe
best_acc, h = train_phase(model, train_loader, val_loader, DEVICE,
    phase_name='S1-A', epochs=s1['phase_a_epochs'],
    head_lr=s1['phase_a_head_lr'], backbone_lr=0.0, freeze=True,
    weight_decay=s1['weight_decay'], warmup_ratio=s1['warmup_ratio'],
    grad_clip=s1['grad_clip'], fp16=s1['fp16'], log_every=s1['log_every'],
    output_dir=s1['output_dir'], best_acc=best_acc)
all_history.extend(h)

# Phase B: full fine-tune
best_acc, h = train_phase(model, train_loader, val_loader, DEVICE,
    phase_name='S1-B', epochs=s1['phase_b_epochs'],
    head_lr=s1['phase_b_head_lr'], backbone_lr=s1['phase_b_backbone_lr'],
    freeze=False,
    weight_decay=s1['weight_decay'], warmup_ratio=s1['warmup_ratio'],
    grad_clip=s1['grad_clip'], fp16=s1['fp16'], log_every=s1['log_every'],
    output_dir=s1['output_dir'], best_acc=best_acc)
all_history.extend(h)

model.save(os.path.join(s1['output_dir'], 'final'))
with open(os.path.join(s1['output_dir'], 'history.json'), 'w') as f:
    json.dump(all_history, f, indent=2)
print(f'\n=== STAGE 1 COMPLETE ===')
print(f'Best val accuracy: {best_acc:.4f}')
print(f'Best checkpoint:   {s1["output_dir"]}/best')


In [ ]:
import matplotlib.pyplot as plt
if all_history:
    fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
    x = range(len(all_history))
    ax.plot(x, [h['accuracy'] for h in all_history], 'o-', label='overall')
    ax.plot(x, [h['male_acc'] for h in all_history], 's--', label='male')
    ax.plot(x, [h['female_acc'] for h in all_history], '^--', label='female')
    ax.set_xlabel('epoch'); ax.set_ylabel('accuracy')
    ax.set_title('Stage 1 val accuracy'); ax.legend(); ax.grid(True, alpha=0.3)
    plt.show()


## 5. Save checkpoint to Google Drive + HuggingFace Hub

In [ ]:
import shutil
stage1_drive = os.path.join(DRIVE_CKPT_DIR, 'stage1')
if os.path.exists(stage1_drive): shutil.rmtree(stage1_drive)
shutil.copytree(os.path.join(s1['output_dir'], 'best'), stage1_drive)
print(f'Copied to Drive: {stage1_drive}')


In [ ]:
# Upload to HuggingFace Hub
# Get your HF token from: huggingface.co → Settings → Access Tokens → Write
"""from huggingface_hub import HfApi
api = HfApi()
import os as _os
if 'HF_TOKEN' in _os.environ:
    api.login(_os.environ['HF_TOKEN'])
else:
    api.login()  # will prompt for token
try:
    api.create_repo(repo_id=CONFIG['hf_repo'], repo_type='model', private=False)
    print(f'Created repo: {CONFIG["hf_repo"]}')
except Exception as e:
    print(f'Repo may already exist: {e}')
api.upload_folder(
    folder_path=os.path.join(s1['output_dir'], 'best'),
    repo_id=CONFIG['hf_repo'],
    repo_type='model',
    commit_message='Stage 1 ViT checkpoint (PETA)'
)
print(f'\nUploaded to: https://huggingface.co/{CONFIG["hf_repo"]}')"""


## Done

Stage 1 checkpoint is now on:
- Google Drive: `/content/drive/MyDrive/IPD_checkpoints/stage1/`
- HuggingFace Hub: `https://huggingface.co/abhshkp/footfall-analysis-vit-stage1`

**Next steps:**
- To test the model: open `test_model.ipynb`
- To train stage 2 on your data: open `train_stage2.ipynb`
